# 从名字到 Bigram 语言模型

这一课是 **Makemore Bigram** 的独立练习。你不会直接复制一个现成模型，而会从名字列表出发，亲手完成 `字符 → 训练对 → 计数 → 概率 → NLL → 神经网络 → 采样` 的完整闭环。

最终你会得到两个能生成新名字的模型：一个从 bigram 次数直接估计概率，另一个用梯度下降学习同一张条件概率表。它们的形式不同，但都在回答同一个问题：**看到当前字符后，下一个字符最可能是什么？**

参考：[Neural Networks: Zero to Hero · Makemore Part 1](https://youtu.be/PaCmpygFfXo) · [课程 Notebook](https://github.com/karpathy/nn-zero-to-hero/blob/master/lectures/makemore/makemore_part1_bigrams.ipynb)

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

torch.manual_seed(2147483647)


In [ ]:
words = '''emma olivia ava isabella sophia charlotte mia amelia harper evelyn abigail emily ella elizabeth camila luna sofia avery mila aria scarlett penelope layla chloe victoria madison eleanor grace nora riley zoey hannah hazel lily ellie violet lillian zoe stella aurora natalie emilia everly leah aubrey willow addison lucy audrey bella nova brooklyn paisley savannah claire skylar isla genesis naomi elena caroline eliana anna maya valentina ruby kennedy ivy ariana aaliyah cora madelyn alice kinsley hailey gabriella allison gianna serenity samantha sarah autumn quinn eva piper sophie sadie delilah josephine neveah adeline arya emery lydia clara vivian madeline peyton julia rylee brielle reagan natalia jade athena maria leilani everleigh liliana melanie mackenzie hadley raelynn kaylee rose arianna isabelle melody eliza lyla katherine aubree adalynn kayla emerson ximena eden ayla kaitlyn calvin liam noah oliver elijah james william benjamin lucas henry alexander mason michael ethan daniel jacob logan jackson levi sebastian mateo jack owen theodore aiden samuel joseph john david wyatt matthew luke asher carter julian grayson leo jayden gabriel isaac lincoln anthony hudson dylan ezra thomas charles christopher jaxon maverick josiah isaiah andrew elias joshua nathan adrian cameron santiago eli aaron ryan angel cooper waylon easton kai christian landon colton roman axel brooks jonathan robert jameson ian everett greyson wesley jeremiah hunter leonardo jordan jose carson bennett silas nicholas parker beau weston austin connor dominic xavier jaxson jace emmett adam declan rowan micah kayden gael river ryder kingston damian sawyer luka evan vincent legend myles harrison august bryson amir giovanni chase diego milo jasper walker jason brayden cole nathaniel george lorenzo zion luis archer enzo jonah thiago theo ayden zachary caleb braxton ashton rhett atlas jude bentley carlos ryker adriel arthur ace tyler jayce max elliot graham kaiden maxwell juan dean matteo malachi ivan elliott jesus emiliano messiah gavin maddox camden hayden leon king jax dean grant abel richard patrick joel timothy miguel edward andres emmanuel griffin alan nicolas peter victor abraham jesse charlie patrick joel richard'''.split()
len(words), words[:8]


## 1. 先把字符变成可以计算的索引

神经网络只处理数字，因此第一步是建立词表。这里的 token 不是单词，而是**单个字符**。我们额外使用 `.` 作为句界符：放在名字前面表示“开始”，放在末尾表示“结束”。同一个 token 同时承担两个角色，能让模型学会首字符和停止条件。

`sorted(set(...))` 会先去重再排序；`enumerate` 同时给出位置和元素。下面的只读示例可以直接运行，观察字符如何得到稳定索引。

In [ ]:
demo_names = ['emma', 'ava', 'mia']
demo_chars = sorted(set(''.join(demo_names)))
demo_stoi = {char: index + 1 for index, char in enumerate(demo_chars)}
demo_stoi['.'] = 0
print(demo_chars)
print(demo_stoi)


### 本节任务

完成 `build_vocabulary(names)`，返回两个普通 Python 字典：

- `stoi: dict[str, int]`：字符到索引；`.` 必须固定为 `0`，其余字符按字母顺序从 `1` 开始。
- `itos: dict[int, str]`：`stoi` 的完整反向映射。
- 不要假定词表一定是 26 个字母；函数必须从传入的 `names` 推导。

```python
build_vocabulary(['anna', 'bob'])[0]
# {'.': 0, 'a': 1, 'b': 2, 'n': 3, 'o': 4}
```

In [ ]:
def build_vocabulary(names: list[str]) -> tuple[dict[str, int], dict[int, str]]:
    """为名字列表建立字符词表，索引 0 固定表示句界符 '.'。

    Example:
        build_vocabulary(['ab'])[0] == {'.': 0, 'a': 1, 'b': 2}
    """
    # TODO: 收集、去重并排序字符，再建立两个互逆字典
    pass

stoi, itos = build_vocabulary(words)
VOCAB_SIZE = len(stoi)
VOCAB_SIZE, stoi


## 2. 把名字拆成输入—目标对

Bigram 模型只看一个字符来预测下一个字符。名字 `emma` 补上边界后是 `.emma.`，因此会产生 `.→e、e→m、m→m、m→a、a→.` 五个训练样本。注意最后的 `a→.` 不是装饰：它教会生成器何时停止。

`torch.tensor(data, dtype=torch.long)` 把 Python 整数列表变成可用于索引的整型 Tensor。官方文档：[torch.tensor](https://docs.pytorch.org/docs/stable/generated/torch.tensor.html)。

In [ ]:
demo_word = 'emma'
demo_tokens = ['.'] + list(demo_word) + ['.']
for current, following in zip(demo_tokens, demo_tokens[1:]):
    print(f'{current!r} -> {following!r}')


### 本节任务

完成 `make_bigram_pairs(names, stoi)`：

- 分别给每个名字的首尾补 `.`，不要跨名字连接。
- 返回 `(inputs, targets)`，二者都是形状 `[N]`、`dtype=torch.long` 的一维 Tensor。
- `inputs[i]` 是当前字符索引，`targets[i]` 是紧随其后的字符索引。
- 一个长度为 `L` 的名字恰好产生 `L + 1` 个训练对。

In [ ]:
def make_bigram_pairs(
    names: list[str],
    stoi: dict[str, int],
) -> tuple[torch.Tensor, torch.Tensor]:
    """把每个名字拆成当前字符索引与下一字符索引。

    Example:
        ['ab'] 应产生 .→a、a→b、b→. 三对。
    """
    # TODO: 分别遍历每个补过边界的名字，收集输入和目标
    pass

xs, ys = make_bigram_pairs(words, stoi)
xs.shape, ys.shape, list(zip(xs[:8].tolist(), ys[:8].tolist()))


## 3. 一张矩阵就是计数语言模型

创建一个 `[V, V]` 的矩阵 `counts`：行索引是当前字符，列索引是下一个字符。`counts[i, j]` 表示训练集中 bigram `i→j` 出现了多少次。第 0 行记录名字如何开始，第 0 列记录名字如何结束。

这里会用到 `torch.zeros(size, dtype=...)` 创建空计数表，再通过二维索引累加。官方文档：[torch.zeros](https://docs.pytorch.org/docs/stable/generated/torch.zeros.html) · [Tensor indexing](https://docs.pytorch.org/docs/stable/tensor_view.html#indexing-slicing-joining-mutating-ops)。

In [ ]:
demo_counts = torch.zeros((3, 3), dtype=torch.int64)
for current, following in [(0, 1), (1, 2), (0, 1)]:
    demo_counts[current, following] += 1
print(demo_counts)
print('0→1 出现次数:', demo_counts[0, 1].item())


### 本节任务

完成 `count_bigrams(inputs, targets, vocab_size)`：

- 返回形状 `[vocab_size, vocab_size]`、`dtype=torch.int64` 的 Tensor。
- 每个 `(inputs[i], targets[i])` 都让对应格子加 `1`，重复 bigram 必须累加。
- 返回矩阵的元素总和应恰好等于训练对数量。

In [ ]:
def count_bigrams(
    inputs: torch.Tensor,
    targets: torch.Tensor,
    vocab_size: int,
) -> torch.Tensor:
    """返回 [V, V] 整数矩阵，其中第 i,j 项是 i→j 的出现次数。"""
    # TODO: 创建整数零矩阵，并累计每一对索引
    pass

counts = count_bigrams(xs, ys, VOCAB_SIZE)
counts.shape, counts.sum().item()


## 4. 从计数得到概率，再用 NLL 给模型打分

把每一行除以该行总和，就得到条件概率 `P(next | current)`。但未见过的 bigram 计数为 0，会让 `log(0)` 变成无穷大；**加一平滑**先给每个格子加 `1`，保证任何转移都有很小但非零的概率。

负对数似然（NLL）只读取模型分配给真实目标的概率：`-log P(target | input)`。越可能的目标产生越小的损失，平均 NLL 越低，模型越好。曲线还能说明为什么这里不能只看预测是否命中：NLL 会区分“给正确答案 51%”和“给正确答案 99%”，也会严厉惩罚把真实目标判成几乎不可能的模型。

这正适合下一个字符预测：每个位置都有一个确定的真实目标，模型输出覆盖整个词表的概率分布；NLL 可逐样本计算、可取平均，并且能通过梯度连续地改进概率。这里会用到广播、`Tensor.sum(dim=..., keepdim=True)`、成对高级索引和 `torch.log`。官方文档：[Tensor.sum](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.sum.html) · [torch.log](https://docs.pytorch.org/docs/stable/generated/torch.log.html)。

In [ ]:
from IPython.display import HTML, display
import math

plot_width, plot_height = 760, 380
left, right, top, bottom = 76, 28, 34, 62
inner_width = plot_width - left - right
inner_height = plot_height - top - bottom
max_loss = 5.0

def x_position(probability: float) -> float:
    return left + probability * inner_width

def y_position(loss: float) -> float:
    return top + (max_loss - loss) / max_loss * inner_height

curve = [(index / 200, -math.log(index / 200)) for index in range(2, 201)]
path = ' '.join(
    f'{"M" if index == 0 else "L"} {x_position(probability):.2f} {y_position(loss):.2f}'
    for index, (probability, loss) in enumerate(curve)
)
grid_lines = ''.join(
    f'<line x1="{left}" y1="{y_position(value):.2f}" x2="{plot_width-right}" y2="{y_position(value):.2f}" stroke="#e4e8df" />'
    f'<text x="{left-12}" y="{y_position(value)+4:.2f}" text-anchor="end" fill="#657064" font-size="12">{value}</text>'
    for value in range(6)
)
x_ticks = ''.join(
    f'<line x1="{x_position(value):.2f}" y1="{plot_height-bottom}" x2="{x_position(value):.2f}" y2="{plot_height-bottom+6}" stroke="#657064" />'
    f'<text x="{x_position(value):.2f}" y="{plot_height-bottom+24}" text-anchor="middle" fill="#657064" font-size="12">{value:g}</text>'
    for value in [0, 0.25, 0.5, 0.75, 1.0]
)
markers = [
    (0.01, '1% → 4.61', 12, 20),
    (0.10, '10% → 2.30', 12, -12),
    (0.50, '50% → 0.69', 12, -12),
    (1.00, '100% → 0', -116, -12),
]
marker_svg = ''.join(
    f'<circle cx="{x_position(probability):.2f}" cy="{y_position(-math.log(probability)):.2f}" r="5" fill="#6d9839" />'
    f'<text x="{x_position(probability)+dx:.2f}" y="{y_position(-math.log(probability))+dy:.2f}" fill="#1d241c" font-size="12" font-weight="600">{label}</text>'
    for probability, label, dx, dy in markers
)
svg = f'''
<div style="max-width:760px;background:#ffffff;padding:12px 8px 4px">
  <svg viewBox="0 0 {plot_width} {plot_height}" role="img" aria-label="真实目标概率从零到一时，负对数似然从无穷大下降到零" style="display:block;width:100%;height:auto">
    <rect x="{left}" y="{top}" width="{inner_width}" height="{inner_height}" fill="#fbfcf8" stroke="#cdd3c8" />
    {grid_lines}{x_ticks}
    <path d="{path}" fill="none" stroke="#6d9839" stroke-width="3" />
    {marker_svg}
    <text x="{plot_width/2}" y="{plot_height-12}" text-anchor="middle" fill="#1d241c" font-size="13">分配给真实下一个字符的概率 p</text>
    <text x="18" y="{plot_height/2}" transform="rotate(-90 18 {plot_height/2})" text-anchor="middle" fill="#1d241c" font-size="13">单样本 NLL = −log(p)</text>
  </svg>
</div>
'''
display(HTML(svg))


In [ ]:
demo_counts = torch.tensor([[0, 2], [1, 0]], dtype=torch.float32)
demo_probs = (demo_counts + 1) / (demo_counts + 1).sum(dim=1, keepdim=True)
demo_x = torch.tensor([0, 1])
demo_y = torch.tensor([1, 0])
demo_selected = demo_probs[demo_x, demo_y]
print(demo_probs)
print('真实目标概率:', demo_selected.tolist())
print('平均 NLL:', (-torch.log(demo_selected)).mean().item())


### 本节任务

在同一个 Cell 中完成两个紧密相关的函数：

1. `normalize_counts(counts, smoothing=1.0) -> torch.Tensor`：把计数转成浮点概率；先加 `smoothing`，再沿 `dim=1` 逐行归一化。每行和必须为 `1`。
2. `negative_log_likelihood(probabilities, inputs, targets) -> torch.Tensor`：用 `probabilities[inputs, targets]` 取出每个真实目标的概率，返回平均 NLL 标量 Tensor。

不要在 NLL 中重新归一化，也不要把 loss 转成 Python float；后面的神经模型还需要相同的度量方式。

In [ ]:
def normalize_counts(counts: torch.Tensor, smoothing: float = 1.0) -> torch.Tensor:
    """加法平滑后逐行归一化，返回 P(next | current)。"""
    # TODO: 转为浮点、加 smoothing，再让每一行除以自身总和
    pass


def negative_log_likelihood(
    probabilities: torch.Tensor,
    inputs: torch.Tensor,
    targets: torch.Tensor,
) -> torch.Tensor:
    """返回所有真实 bigram 概率的平均负对数。"""
    # TODO: 成对索引真实目标概率，取 log、加负号并求平均
    pass

count_probabilities = normalize_counts(counts, smoothing=1.0)
count_nll = negative_log_likelihood(count_probabilities, xs, ys)
count_nll


## 5. 神经网络版其实也在学习一张 `[V, V]` 表

`F.one_hot(inputs, num_classes=V)` 把字符索引变成 one-hot 行向量。它乘以 `[V, V]` 权重矩阵时，只会选出对应的那一行；这和上一节按当前字符选择计数概率行是同一件事。区别在于，这张表现在存的是可学习的 **logits**，而不是人工统计的概率。

`nn.Parameter` 会把 Tensor 登记为模型参数，`model.parameters()` 才能把它交给优化器。官方文档：[F.one_hot](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.one_hot.html) · [nn.Parameter](https://docs.pytorch.org/docs/stable/generated/torch.nn.parameter.Parameter.html) · [nn.Module](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)。

In [ ]:
demo_ids = torch.tensor([2, 0])
demo_one_hot = F.one_hot(demo_ids, num_classes=3).float()
demo_weight = torch.tensor([[10., 11.], [20., 21.], [30., 31.]])
print(demo_one_hot)
print(demo_one_hot @ demo_weight)
print('等于直接选行:', demo_weight[demo_ids])


### 本节任务

完成 `NeuralBigram(nn.Module)`：

- `__init__(vocab_size)` 创建唯一参数 `weight`，形状严格为 `[V, V]`；可用 `torch.randn(vocab_size, vocab_size)` 初始化。
- `forward(inputs)` 接收 `[N]` 的字符索引，先做 `F.one_hot(..., num_classes=V).float()`，再乘 `weight`。
- 返回 `[N, V]` 的原始 logits；不要在模型内部 softmax，也不要 `detach`。

当输入索引为 `i` 时，输出必须恰好等于 `weight[i]`。

In [ ]:
class NeuralBigram(nn.Module):
    """用一个 [V, V] 参数矩阵预测下一个字符的 logits。"""

    def __init__(self, vocab_size: int) -> None:
        super().__init__()
        self.vocab_size = vocab_size
        # TODO: 创建形状 [V, V] 的 nn.Parameter，属性名必须是 weight

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        """把 [N] 字符索引映射为 [N, V] logits。"""
        # TODO: one-hot 编码后与 weight 做矩阵乘法
        pass


neural_model = NeuralBigram(VOCAB_SIZE)
initial_logits = neural_model(xs)
initial_logits.shape


## 6. 用交叉熵和 SGD 学会这张表

`F.cross_entropy(logits, targets)` 内部完成 `log_softmax + NLL`，所以它直接接收 logits，不需要你先做 softmax。一次训练迭代有固定的五步：清空旧梯度、前向计算、计算 loss、反向传播、更新参数。

`optimizer.zero_grad()` 很关键，因为 PyTorch 默认会累加梯度；`loss.backward()` 根据计算图求导；`optimizer.step()` 使用当前梯度更新 `nn.Parameter`。官方文档：[F.cross_entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html) · [torch.optim.SGD](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html) · [Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)。

In [ ]:
demo_parameter = nn.Parameter(torch.tensor([0.0]))
demo_optimizer = torch.optim.SGD([demo_parameter], lr=0.1)
demo_optimizer.zero_grad()
demo_loss = (demo_parameter - 3.0).pow(2).sum()
demo_loss.backward()
print('梯度:', demo_parameter.grad.item())
demo_optimizer.step()
print('更新后的参数:', demo_parameter.item())


### 本节任务

完成 `train_bigram(model, inputs, targets, steps, learning_rate)`：

- 创建 `torch.optim.SGD(model.parameters(), lr=learning_rate)`。
- 每轮按 `zero_grad → model(inputs) → F.cross_entropy → backward → step` 的顺序执行。
- 每轮将 `float(loss.detach())` 追加到列表，最终返回长度等于 `steps` 的 `list[float]`。
- 不要重新创建模型，不要替换传入的参数，也不要只返回最后一个 loss。

In [ ]:
def train_bigram(
    model: NeuralBigram,
    inputs: torch.Tensor,
    targets: torch.Tensor,
    steps: int = 100,
    learning_rate: float = 10.0,
) -> list[float]:
    """用全量数据和 SGD 训练模型，返回每一步的交叉熵。"""
    # TODO: 创建优化器并实现完整训练循环
    pass

loss_history = train_bigram(neural_model, xs, ys, steps=100, learning_rate=10.0)
loss_history[0], loss_history[-1]


## 7. 自回归采样：把概率表变回名字

生成从句界符 `.`（索引 0）开始。每一步读取“当前字符”对应的概率行，用 `torch.multinomial` 抽出下一个字符，再把它作为下一轮输入；抽到 0 就结束。这就是最小的自回归循环。

温度会在采样前重塑分布：`softmax(log(p) / temperature)`。温度低于 1 更保守，高于 1 更随机。固定 `torch.Generator` 的 seed 可以让实验可复现。官方文档：[torch.multinomial](https://docs.pytorch.org/docs/stable/generated/torch.multinomial.html) · [torch.Generator](https://docs.pytorch.org/docs/stable/generated/torch.Generator.html) · [torch.softmax](https://docs.pytorch.org/docs/stable/generated/torch.softmax.html)。

In [ ]:
demo_generator = torch.Generator().manual_seed(7)
demo_distribution = torch.tensor([0.1, 0.7, 0.2])
demo_draws = [
    torch.multinomial(demo_distribution, 1, generator=demo_generator).item()
    for _ in range(8)
]
demo_draws


### 本节任务

完成 `sample_names(probabilities, itos, num_samples, seed, temperature, max_length)`：

- `probabilities` 是 `[V, V]` 概率表；行是当前字符，列是下一个字符。
- 每个名字从索引 `0` 开始；抽到 `0` 立即停止，且不要把 `.` 放进结果字符串。
- 每一步先对当前行做 `softmax(log(row.clamp_min(1e-12)) / temperature)`，再 `torch.multinomial(..., generator=generator)`。
- 返回恰好 `num_samples` 个字符串；`max_length` 是防止异常概率表永不结束的上限。
- 相同参数与 seed 必须产生相同结果。

In [ ]:
def sample_names(
    probabilities: torch.Tensor,
    itos: dict[int, str],
    num_samples: int = 5,
    seed: int = 42,
    temperature: float = 1.0,
    max_length: int = 20,
) -> list[str]:
    """从索引 0 开始逐字符采样，抽到 0 时结束名字。"""
    # TODO: 创建可复现的 generator，并实现自回归采样循环
    pass

count_samples = sample_names(count_probabilities, itos, num_samples=8, seed=42)
count_samples


## 最终运行：比较两个模型

计数模型把经验次数直接归一化；神经模型通过交叉熵和梯度下降学习 logits。把神经网络的 `weight` 沿每一行 softmax 后，它也变成 `[V, V]` 概率表，因此两者可以使用同一个 NLL 和同一个采样器。

运行下面的 Cell，确认整章代码已经连成一个系统。然后尝试改变 `smoothing`、`temperature` 或 `seed`，观察 NLL 与生成结果如何变化。

In [ ]:
with torch.no_grad():
    neural_probabilities = torch.softmax(neural_model.weight, dim=1)
    neural_nll = negative_log_likelihood(neural_probabilities, xs, ys)

print(f'训练对数量: {len(xs)}，词表大小: {VOCAB_SIZE}')
print(f'计数模型 NLL: {count_nll.item():.4f}')
print(f'神经模型 NLL: {neural_nll.item():.4f}')
print('\n计数模型生成:')
print(sample_names(count_probabilities, itos, num_samples=10, seed=7))
print('\n神经模型生成:')
print(sample_names(neural_probabilities, itos, num_samples=10, seed=7))
